# Notebook 05 — Final Report
**Project:** Customer Churn Prediction · Olist Brazilian E-Commerce
**Author:** Farah Aulia Kirana · dikerjakan 2026
**Stack:** Python · scikit-learn · XGBoost · SHAP · FastAPI
**Model v1 (churn):** XGBoost · ROC-AUC 0.8207 · CV AUC 0.8213 ± 0.0087
**Model v2 (repeat purchase, snapshot):** RFM Logistic Regression / XGBoost · backtest 3 periode: lift top 1% 6.1–6.7× (RFM LogReg)

---

> Notebook ini adalah **ringkasan eksekutif end-to-end** dari seluruh proyek.
> Dirancang untuk dibaca oleh siapapun — teknikal maupun non-teknikal —
> tanpa perlu membuka NB01–NB04 terlebih dahulu.
>
> Semua angka di laporan ini berasal dari hasil eksekusi NB01–NB04 terbaru dan identik dengan `outputs/metrics.json` yang dihasilkan `train_pipeline.py`.

---

## Daftar Isi
1. [Problem Statement & Konteks Bisnis](#1-problem-statement)
2. [Dataset & Pipeline](#2-dataset--pipeline)
3. [Temuan Utama EDA](#3-temuan-utama-eda)
4. [Feature Engineering — Keputusan Desain](#4-feature-engineering)
5. [Modeling & Evaluasi](#5-modeling--evaluasi)
6. [Model Interpretability — SHAP](#6-shap)
7. [Business Recommendations](#7-business-recommendations)
8. [Keterbatasan & Next Steps](#8-keterbatasan)
9. [Kesimpulan Eksekutif](#9-kesimpulan)

---
## 1. Problem Statement

### Konteks Bisnis

Olist adalah marketplace e-commerce Brazil yang menghubungkan ribuan penjual kecil dengan jutaan konsumen di seluruh negeri. Dataset publik Olist mencakup **~100 ribu transaksi nyata** dari September 2016 hingga Oktober 2018 — salah satu dataset e-commerce paling lengkap yang tersedia secara publik.

### Masalah

Data menunjukkan bahwa **96.9% customer Olist hanya melakukan satu kali pembelian** dan tidak pernah kembali. Tanpa kemampuan mengidentifikasi siapa yang berpotensi churn — dan lebih penting lagi, *mengapa* mereka churn — tim marketing tidak bisa mengalokasikan budget retensi secara efektif.

### Tujuan

| # | Tujuan | Ukuran Keberhasilan | Status |
|---|---|---|---|
| 1 | Prediksi customer yang akan churn sebelum mereka pergi | ROC-AUC ≥ 0.80 | ✅ 0.8207 — dengan catatan validitas label (Section 8) |
| 2 | Identifikasi *mengapa* customer churn — bukan hanya *siapa* | SHAP-based interpretation | ✅ Dengan batasan: SHAP menjelaskan model, bukan sebab-akibat |
| 3 | Terjemahkan hasil model ke rekomendasi bisnis yang actionable | Minimal 3 rekomendasi konkret | ✅ Disertai desain A/B test |
| 4 | Bangun pipeline yang bersih dan bebas data leakage | AUC pre vs post leakage fix terdokumentasi | ⚠️ Leakage `first_order_month` diatasi; leakage waktu lewat fitur lain terdeteksi lewat uji placebo |

### Definisi Churn

Customer dikategorikan **churn** jika tidak melakukan order dalam **90 hari** terakhir sebelum tanggal referensi (17 Oktober 2018, order terakhir di dataset).

Alasan memilih 90 hari:
- Siklus repeat purchase e-commerce umum berkisar 30–120 hari
- 90 hari cukup membedakan *'belum beli lagi'* vs *'sudah pergi'* untuk produk campuran seperti Olist

> ⚠️ Karena tanggal referensinya tetap, definisi ini identik dengan *"order terakhir sebelum 19 Juli 2018"*. Konsekuensinya dibahas di Section 5.6 dan 8.

---
## 2. Dataset & Pipeline

### Dataset

**Sumber:** [Olist Brazilian E-Commerce Public Dataset](https://www.kaggle.com/olistbr/brazilian-ecommerce) (Kaggle)
**Periode data:** September 2016 – Oktober 2018 (transaksi praktis berhenti akhir Agustus 2018)  
**Catatan:** dataset historis yang dirilis publik oleh Olist — tahun 2016–2018 adalah periode transaksi di dalam data, bukan waktu proyek dikerjakan (2026)
**Lisensi:** CC BY-NC-SA 4.0

| Tabel | Baris | Deskripsi |
|---|---|---|
| `orders` | 99,441 | Transaksi utama — ID, status, timestamp |
| `customers` | 99,441 | Profil & lokasi customer |
| `order_items` | 112,650 | Detail item per order (harga, freight) |
| `payments` | 103,886 | Metode & nilai pembayaran |
| `reviews` | 99,224 | Rating & komentar customer |
| `products` | 32,951 | Katalog produk & kategori |
| `category_translation` | 71 | Terjemahan nama kategori ke Bahasa Inggris |

### Pipeline End-to-End

```
📁 data/raw/          ← 7 CSV mentah dari Kaggle
     │
     ▼
📓 01_data_loading.ipynb
     └─ Merge 7 tabel → Validasi (0 duplikat, semua 99,441 order terwakili)
     └─ Output: data/processed/master.csv  (113,425 baris × 23 kolom, grain = item)
     │
     ▼
📓 02_eda.ipynb
     └─ Definisi churn (90 hari) → RFM analysis (dedup per order) → Freight quartile
     └─ Cek confounding ongkir vs lokasi & waktu → Dokumentasi right-censoring
     └─ Output: data/processed/rfm_labeled.csv  (96,096 customers)
     │
     ▼
📓 03_feature_engineering.ipynb
     └─ Log-transform → Agregasi dua tahap (order & item) → Fitur derived → OHE
     └─ Deteksi multikolinearitas → export features_lr.csv untuk LR
     └─ Output: features.csv (RF/XGB)  |  features_lr.csv (Logistic Reg.)
     │
     ▼
📓 04_modeling.ipynb
     └─ Split 60/20/20 → Benchmark 3 model → Deteksi & fix leakage
     └─ XGBoost challenger → SHAP → Threshold tuning (validation) → Lift → Uji placebo
     └─ Output: outputs/model_comparison_final.csv  |  outputs/figures/
     │
     ▼
📓 05_final_report.ipynb  ← Anda sedang membaca ini
     │
     ▼
📓 06_snapshot_redesign.ipynb
     └─ Desain ulang label: snapshot T, fitur < T, label = repeat di [T, T+90)
     └─ Out-of-time test (Jun 2018) + baseline + uji signifikansi bootstrap
     └─ Backtest 3 periode test (Des 2017, Mar 2018, Jun 2018)

⚙️ train_pipeline.py  → menghasilkan outputs/model_xgb.pkl + metrics.json untuk API
                         (logika identik dengan NB03–NB04)
⚙️ train_snapshot_pipeline.py → model_snapshot_*.pkl + metrics_snapshot.json (= NB06)
```

---
## 3. Temuan Utama EDA

### 3.1 Distribusi Target

| Label | Jumlah | Persentase |
|---|---|---|
| Churn (1) | 86,432 | 89.9% |
| Non-Churn (0) | 9,664 | 10.1% |

> ⚠️ **Right-Censoring:** Churn rate 89.9% mengandung artefak dataset. Customer yang order terakhirnya dekat cutoff Oktober 2018 belum punya cukup waktu untuk repeat order. Angka ini adalah *upper bound*, bukan gambaran bisnis sesungguhnya.

### 3.2 RFM Analysis

| Metrik | Churn | Non-Churn | Kesimpulan |
|---|---|---|---|
| Avg Recency | 312 hari | 72 hari | Pembeda terkuat — tetapi by definition (label dibuat dari recency) |
| Avg Frequency | 1.03 order | 1.04 order | Hampir tidak membedakan |
| Avg Monetary | BRL 166.51 | BRL 167.30 | Tidak membedakan |

**Takeaway:** Frequency dan monetary hampir tidak membedakan kedua kelas — karena **96.9% customer hanya order 1 kali**. Bahkan **96.3% customer non-churn juga one-time buyer** yang kebetulan baru berbelanja. Ini menggeser fokus dari *retention* ke *activation*.

> **Koreksi perhitungan:** `master.csv` ber-grain item, sementara `payment_value` bernilai per order. Versi sebelumnya menjumlahkan payment di level item sehingga monetary rata-rata tercatat BRL 213 — sekitar 28% lebih tinggi dari nilai sebenarnya (BRL 166.59).

### 3.3 Churn Rate by Freight Quartile

| Freight Quartile | Avg Freight | Churn Rate |
|---|---|---|
| Q1 — ongkir rendah | < BRL 13.37 | 89.2% |
| Q2 | BRL 13.37–16.40 | **93.3%** |
| Q3 | BRL 16.40–21.22 | 89.5% |
| Q4 — ongkir tinggi | > BRL 21.22 | **87.7%** |

Hipotesis awal "ongkir tinggi → churn lebih tinggi" **tidak didukung data**. Hubungannya tidak monoton: churn tertinggi di Q2, terendah justru di Q4.

**Cek confounding:**
- **Lokasi:** customer SP membayar ongkir lebih murah (BRL 15.32 vs 23.77) dan churn-nya lebih rendah (88.3% vs 91.1%). Tetapi di dalam SP maupun non-SP, Q4 tetap punya churn terendah — jadi pola ini bukan sekadar efek lokasi
- **Waktu:** rata-rata ongkir bulanan berkorelasi **−0.44** dengan churn rate bulanan. Ongkir ikut membawa informasi *kapan* customer berbelanja

### 3.4 Temuan EDA Lain

- **Review score hanya sedikit membedakan churn** — churn rate turun dari 92.7% (score 1) ke 88.9% (score 5)
- **Tren order tumbuh pesat dari Sep 2016 ke Nov 2017** — lalu stabil di 2018
- **Top kategori by revenue item: `health_beauty`, `watches_gifts`, `bed_bath_table`** — `health_beauty` dan `bed_bath_table` adalah produk habis pakai dengan potensi repeat purchase alami

---
## 4. Feature Engineering — Keputusan Desain

### 4.1 Ringkasan Fitur

| Kategori | Fitur | Jumlah |
|---|---|---|
| RFM + log-transform | `recency`, `frequency`, `monetary`, `log_recency`, `log_monetary`, `log_frequency` | 6 |
| Agregasi pembayaran (level order) | `avg_payment_value`, `total_payment_value`, `avg_installments`, `max_installments` | 4 |
| Agregasi harga & ongkir (level item) | `avg_price`, `total_price`, `avg_freight` | 3 |
| Agregasi review (level order) | `avg_review_score`, `min_review_score`, `review_count` | 3 |
| Agregasi order | `total_items`, `total_orders` | 2 |
| Fitur derived | `customer_lifetime_days`, `avg_order_value`, `freight_ratio`, `review_gap`, `first_order_month`, `first_order_dayofweek` | 6 |
| OHE Geografis | `customer_state_*` (27 state) | 27 |
| OHE Payment type | `dominant_payment_type_*` | 6 |
| OHE Kategori produk | `dominant_category_*` (top 10 + other) | 11 |
| **Total kolom `features.csv`** (termasuk ID & target) | | **70** |

Setelah `recency`, `log_recency`, `first_order_month`, dan `first_order_dayofweek` di-drop, model final memakai **64 fitur**.

### 4.2 Keputusan Desain Penting

| Keputusan | Pilihan | Alasan |
|---|---|---|
| Unit analisis | `customer_unique_id` | Target = customer, bukan order |
| Agregasi | Dua tahap: level order & level item | Payment & review berulang di setiap item — tanpa dedup terhitung ganda |
| `total_items` | `count` item | `order_item_id` adalah nomor urut, bukan jumlah |
| RFM transform | `log1p` | Distribusi right-skewed ekstrem; `log1p` aman untuk nilai 0 |
| Missing values | Median imputation di dalam Pipeline | Median dihitung hanya dari data train — tidak ada kebocoran statistik test |
| Kategori produk | Top 10 + `other` | Mencegah OHE sparse dari 70+ kategori |
| Drop dari feature matrix | `recency`, `log_recency` | Target didefinisikan dari recency — direct leakage |
| Drop di NB04 | `first_order_month`, `first_order_dayofweek` | Artefak right-censoring; terbukti leaky |

### 4.3 Dua Feature Matrix — Satu Keputusan Penting

**Masalah:** 23 pasangan fitur memiliki |r| > 0.9, termasuk r = 1.0 (`monetary` ↔ `total_payment_value`, `frequency` ↔ `total_orders`, `avg_payment_value` ↔ `avg_order_value`).

**Solusi:** Export dua feature set berbeda:

| File | Dipakai untuk | Perlakuan multikolinearitas |
|---|---|---|
| `features.csv` (70 kolom) | Random Forest, XGBoost | Dibiarkan — tree handles ini |
| `features_lr.csv` (61 kolom) | Logistic Regression | 9 kolom redundan di-drop |

Ini **memastikan perbandingan antar model fair** — AUC LR yang rendah mencerminkan keterbatasan algoritma, bukan kesalahan setup.

---
## 5. Modeling & Evaluasi

### 5.1 Setup

| Aspek | Keputusan | Alasan |
|---|---|---|
| Split | 60/20/20 stratified (train / validation / test) | Threshold dipilih di validation; test hanya untuk evaluasi akhir |
| Metric utama | **ROC-AUC** & **lift per decile** | AUC tidak terpengaruh threshold; lift menjawab pertanyaan targeting |
| Metric pendukung | F1-Score weighted | Mempertimbangkan kedua kelas |
| Metric diabaikan | Accuracy | Menyesatkan — model yang selalu prediksi 'churn' dapat 89.9% |
| Imbalance handling | `class_weight='balanced'` (LR/DT/RF); XGBoost tanpa reweighting | XGBoost balanced (spw = 0.11) diuji dan sedikit lebih buruk |
| Validasi | 5-fold Stratified CV di dev set (train + validation) | Estimasi performa yang stabil tanpa menyentuh test set |

### 5.2 Deteksi & Penanganan Data Leakage

> **Ini adalah temuan terpenting pertama dalam proyek ini.**

Model awal menghasilkan AUC 0.9728 — performa yang mencurigakan untuk data yang didominasi one-time buyer. Investigasi feature importance mengungkap penyebabnya:

```
Feature importance pre-fix (Random Forest):
  first_order_month      : 0.550  ← MENDOMINASI semua fitur lain
  avg_freight            : 0.078
  freight_ratio          : 0.034
  ...
```

Analisis churn rate per bulan (kalender, gabungan 2016–2018) mengkonfirmasi leakage:

| first_order_month | Churn Rate | Keterangan |
|---|---|---|
| Jan–Jun, Sep–Des | ~99.7% | Hampir semua churn |
| Jul | 68.1% | Mulai turun |
| Agu | 39.9% | Customer yang order pertama kali di Jul–Agu 2018 belum sempat terlabel churn |

Model 'belajar' bahwa customer yang first order-nya di bulan-bulan dekat cutoff pasti non-churn — bukan karena pola perilaku, tapi karena dataset berakhir. Setelah `first_order_month` dan `first_order_dayofweek` di-drop:

| Model | AUC dengan Leakage | AUC Tanpa Leakage | Δ |
|---|---|---|---|
| Logistic Regression | 0.7127 | 0.6021 | −0.111 |
| Decision Tree | 0.9633 | 0.7748 | −0.189 |
| Random Forest | 0.9728 | 0.8005 | −0.172 |

### 5.3 Perbandingan Model Final (Clean — No Leakage)

| Model | ROC-AUC (test) | CV AUC (dev) | F1 (thr 0.5) |
|---|---|---|---|
| **XGBoost** 🏆 | **0.8207** | **0.8213 ± 0.0087** | 0.8639 |
| Random Forest | 0.8005 | 0.8025 ± 0.0058 | 0.8800 |
| Decision Tree | 0.7748 | 0.7863 ± 0.0131 | 0.7697 |
| Logistic Regression | 0.6021 | 0.6122 ± 0.0080 | 0.6655 |

**Mengapa XGBoost menang:** AUC dan CV AUC tertinggi — paling baik mengurutkan customer berdasarkan risiko. F1 di threshold 0.5 tidak dipakai untuk memilih model karena threshold 0.5 tidak relevan untuk model ini (lihat 5.4).

### 5.4 Threshold Tuning (di Validation Set)

| Metode | Threshold | Customer ditandai churn | Non-churn recall |
|---|---|---|---|
| Max F1 kelas churn (versi lama) | 0.6426 | 96.8% | 17.8% |
| **Youden's J (TPR − FPR)** | **0.9004** | **64.7%** | **77.3%** |

Memaksimalkan F1 kelas churn — kelas **mayoritas** — membuat model menandai hampir semua customer sebagai churn. Versi sebelumnya (threshold 0.470, churn recall 99.2%) berasal dari pendekatan ini dan dihitung di test set. Youden's J menimbang kedua kelas setara.

### 5.5 Evaluasi Akhir di Test Set

| Metrik | Threshold 0.5 | Youden (0.9004) |
|---|---|---|
| Churn recall | 99.5% | 69.0% |
| Non-churn recall | 6.3% | **78.7%** |
| Precision churn (baseline "semua churn" = 89.9%) | ~90% | **96.7%** |

**Lift per decile:**

| Decile skor | Churn rate | Customer kembali | Lift (kembali) |
|---|---|---|---|
| 1 (risiko tertinggi) | 99.5% | 0.5% | 0.05× |
| 9 | 79.9% | 20.1% | 2.00× |
| 10 (risiko terendah) | 59.6% | **40.4%** | **4.02×** |

Dengan base rate churn 90%, lift maksimum di sisi churn hanya 1.11× (model mencapai 1.107×). Sisi yang informatif adalah sebaliknya: 10% skor terendah berisi 4× lebih banyak customer yang kembali.

### 5.6 Uji Placebo — Temuan Terpenting Kedua

Karena label churn identik dengan "order terakhir sebelum 19 Juli 2018", model XGBoost yang sama dilatih untuk menebak jendela waktu pembelian **lain** (label palsu):

| Jendela order terakhir | ROC-AUC |
|---|---|
| **Churn asli** (19 Jul – 17 Okt 2018) | **0.826** |
| Placebo A (20 Apr – 19 Jul 2018) | 0.804 |
| Placebo C (20 Jul – 18 Okt 2017) | 0.738 |
| Placebo B (20 Jan – 20 Apr 2018) | 0.675 |

Fitur yang sama mampu menebak *kapan* customer berbelanja dengan AUC 0.68–0.80. AUC churn hanya sedikit lebih tinggi dari placebo terdekat. **Sebagian besar kemampuan model berasal dari mengenali periode pembelian**, bukan niat customer untuk kembali. Decile dengan "40.4% customer kembali" pada dasarnya berisi customer yang baru berbelanja di periode akhir dataset.

---
## 6. Model Interpretability — SHAP

Feature importance XGBoost hanya menjawab *'fitur mana yang berguna untuk model'*, bukan arah pengaruhnya. SHAP (SHapley Additive exPlanations) menjawab: **fitur ini menaikkan atau menurunkan prediksi risiko churn, sebesar apa, untuk siapa.**

### 6.1 Top SHAP Insights

| Fitur | Arah (SHAP) | Mean \|SHAP\| | Interpretasi |
|---|---|---|---|
| `avg_freight` | Tinggi → churn **↓** (tidak monoton, puncak risiko di Q2) | 0.668 | Berlawanan dengan hipotesis awal; sebagian kemungkinan efek waktu |
| `customer_state_SP` | SP = True → churn ↓ | 0.216 | Customer São Paulo berisiko lebih rendah — dekat pusat logistik |
| `avg_review_score` | Tinggi → churn ↓ | 0.067 | Kepuasan sedikit berpengaruh |
| `dominant_payment_type_debit_card` | Debit = True → churn ↓ | 0.039 | Kemungkinan proksi waktu: porsi debit melonjak di Q3 2018 |
| `dominant_category_toys` | Toys = True → churn ↑ | 0.033 | Pembeli mainan cenderung one-time buyer musiman |

### 6.2 Temuan Kritis: SHAP vs Feature Importance vs Kausalitas

**`dominant_payment_type_debit_card`** adalah fitur #1 di feature importance XGBoost (0.081). SHAP menunjukkan arahnya: pengguna debit justru **berisiko lebih rendah**. Feature importance saja tidak bisa memberi tahu ini.

Namun SHAP juga punya batas: SHAP menjelaskan **perilaku model**, bukan **sebab-akibat**. Porsi pembayaran debit naik dari ~1% menjadi 4% di kuartal ke-3 2018 — periode yang hampir pasti terlabel non-churn. Jadi "pengguna debit lebih loyal" kemungkinan besar adalah artefak waktu, bukan perilaku.

| Aspek | Feature Importance | SHAP | A/B Test |
|---|---|---|---|
| Menunjukkan arah pengaruh | ❌ Tidak | ✅ Ya | ✅ Ya |
| Per-instance explanation | ❌ Tidak | ✅ Ya | ❌ Tidak |
| Membuktikan sebab-akibat | ❌ Tidak | ❌ Tidak | ✅ Ya |

---
## 7. Business Recommendations

Rekomendasi disusun berdasarkan **kekuatan bukti**. Karena uji placebo menunjukkan model banyak menangkap informasi waktu, rekomendasi yang hanya bersandar pada SHAP diperlakukan sebagai **hipotesis yang perlu diuji**, bukan kesimpulan.

### Prioritas 1 — Second Purchase Activation
> *Voucher 15% untuk order kedua, dikirim 14 hari setelah first order*

**Dasar (bukti kuat, langsung dari data):** 96.9% customer hanya order 1 kali, dan 96.3% customer non-churn pun one-time buyer. Ini bukan masalah *retention* — ini masalah *activation*.

**Implementasi:** Trigger email/notif otomatis D+14 post first order, dengan holdout group 50% sebagai kontrol. Metrik utama: repeat purchase rate 90 hari.

**Target:** Semua customer baru — bukan hanya yang diprediksi churn.

---

### Prioritas 2 — Uji Free Shipping sebagai Eksperimen
> *'Gratis ongkir untuk order kedua di atas BRL 150'*

**Dasar (bukti lemah):** EDA dan SHAP **tidak** menunjukkan bahwa ongkir tinggi menaikkan churn — Q4 ongkir justru punya churn terendah. Namun ongkir tetap faktor biaya yang masuk akal untuk diuji, terutama untuk customer di luar SP.

**Implementasi:** A/B test — 50% customer baru mendapat voucher gratis ongkir untuk order kedua. Ukur repeat purchase rate 90 hari **dan** biaya subsidi per customer yang kembali.

---

### Prioritas 3 — Segmentasi Geografis
> *Prioritaskan eksperimen di customer non-SP*

**Dasar:** Customer non-SP punya churn rate lebih tinggi (91.1% vs 88.3%) dan SHAP `customer_state_SP` konsisten negatif. Ini sinyal yang relatif stabil.

**Implementasi:** Gunakan wilayah sebagai strata dalam A/B test Prioritas 1 & 2, agar efek program bisa dibandingkan antara SP dan non-SP.

---

### Prioritas 4 — Segmentasi Kategori Produk
> *Arahkan insentif next-purchase ke kategori habis pakai*

**Dasar:** `dominant_category_toys` menaikkan prediksi churn (SHAP positif) — pembeli mainan cenderung musiman. `health_beauty` dan `bed_bath_table` adalah kategori dengan revenue besar dan potensi repeat purchase alami.

**Implementasi:** Prioritaskan budget voucher untuk pembeli kategori habis pakai; ukur apakah repeat purchase rate-nya memang lebih tinggi.

---

### Yang **tidak** direkomendasikan

**Cashback khusus debit card.** Versi sebelumnya merekomendasikan ini berdasarkan SHAP. Setelah dicek, porsi pembayaran debit melonjak justru di periode yang hampir pasti terlabel non-churn (Q3 2018) — sinyal ini kemungkinan besar artefak waktu, bukan perilaku loyal.

---

### Ukuran Dampak (Skenario)

Angka di bawah adalah **skenario**, bukan prediksi — ukuran efek sebenarnya hanya bisa diketahui dari A/B test.

Asumsi: rata-rata nilai order BRL 161 (setelah koreksi double counting), satu order tambahan per customer yang kembali, **belum dikurangi biaya voucher/subsidi**.

| Customer yang kembali per 10,000 customer yang ditargetkan | Customer | Revenue gross |
|---|---|---|
| 2% | 200 | ~BRL 32K |
| 5% | 500 | ~BRL 81K |
| 10% | 1,000 | ~BRL 161K |

> Versi sebelumnya mencantumkan "BRL 361K per cohort" dengan avg order value BRL 213 (masih terhitung ganda) dan asumsi conversion 10% tanpa dasar. Angka tersebut sudah tidak dipakai.

---
## 8. Keterbatasan & Next Steps

### 8.1 Keterbatasan yang Perlu Diketahui

**Definisi label = tanggal order terakhir (Kritis)**
`is_churned` identik dengan "order terakhir sebelum 19 Juli 2018". Uji placebo menunjukkan fitur yang sama bisa menebak jendela waktu pembelian lain dengan AUC 0.68–0.80. Artinya model saat ini lebih tepat dibaca sebagai **pengenal customer yang baru berbelanja**, bukan prediktor niat kembali. Model belum layak dipakai untuk memutuskan siapa yang perlu intervensi.

**Right-Censoring (Kritis)**
Churn rate 89.9% mengandung artefak cutoff Oktober 2018. Customer yang order terakhirnya dekat cutoff belum punya waktu untuk kembali.

**Random Split vs Temporal Split (Medium)**
Split acak membiarkan model melihat periode yang sama di train dan test. Desain snapshot atau temporal split memberi evaluasi yang lebih realistis.

**Dominance One-Time Buyer (Struktural)**
96.9% customer hanya order 1 kali — sinyal perilaku per customer sangat terbatas.

### 8.2 Next Steps

| Prioritas | Item | Estimasi Effort |
|---|---|---|
| ✅ Selesai | Desain ulang label dengan snapshot — lihat 8.3 dan NB06 | — |
| 🔴 High | A/B test second purchase voucher (Prioritas 1) | 2 minggu setup, 90 hari observasi |
| ✅ Selesai | Backtesting model snapshot di 3 periode test — lihat 8.3 | — |
| 🟡 Medium | Migrasi API ke model snapshot (skema request baru: `recency_days`, `tenure_days`, dll.) | 2–3 hari |
| 🟡 Medium | Ganti deteksi drift berbasis selisih mean dengan PSI / uji KS | 1 hari |
| 🟡 Medium | Hyperparameter tuning XGBoost (Optuna) setelah label diperbaiki | 2–3 hari |
| 🟢 Low | Uplift modeling untuk menentukan siapa yang benar-benar terpengaruh voucher | 1–2 minggu |
| 🟢 Low | Survival Analysis untuk memodelkan *kapan* customer melakukan pembelian berikutnya | 1–2 minggu |

### 8.3 Hasil Desain Snapshot (NB06)

Label didesain ulang: untuk tanggal snapshot T, fitur hanya dari data sebelum T dan label = customer order lagi dalam 90 hari setelah T. Train di snapshot Jun/Sep/Des 2017, validation Mar 2018, test **out-of-time** Jun 2018 (77,808 customer, 440 repeater, base rate 0.57%).

| Model | ROC-AUC | PR-AUC | Lift top 1% | Lift top 10% |
|---|---|---|---|---|
| Recency rule (hanya timing) | 0.582 | 0.0080 | 1.36× | 1.86× |
| RFM Logistic Regression | 0.602 | 0.0188 | 6.36× | 2.25× |
| XGBoost | 0.619 | 0.0192 | 6.82× | 2.55× |

- **Sinyalnya nyata tetapi lemah:** XGBoost signifikan lebih baik dari aturan recency (selisih PR-AUC +0.011, CI 95% 0.004–0.022), jadi model menangkap lebih dari sekadar timing
- **Model sederhana sudah cukup:** XGBoost tidak signifikan lebih baik dari RFM Logistic Regression (CI memuat 0) → LogReg direkomendasikan sebagai model produksi pertama
- **Sinyal utama:** pernah order lebih dari sekali (repeat rate 3.4× base), recency yang baru, dan review yang tinggi. Ongkir berpengaruh kecil
- **Backtest 3 periode (Des 2017, Mar 2018, Jun 2018):** XGBoost signifikan mengalahkan aturan recency di 3/3 periode, tetapi tidak pernah signifikan lebih baik dari RFM LogReg. LogReg lebih tinggi di 2/3 periode dan lebih stabil (lift top 1% 6.1–6.7×, rata-rata 6.4×; XGBoost 4.8–6.8×). Jun 2018 ternyata periode terbaik XGBoost
- **Catatan data:** order praktis berhenti di akhir Agustus 2018, sehingga "90 hari terakhir" di desain churn lama hanya berisi ~43 hari data

---
## 9. Kesimpulan Eksekutif

### Scorecard

| Aspek | Hasil |
|---|---|
| **Champion Model** | XGBoost (tanpa class reweighting) |
| **ROC-AUC (test)** | 0.8207 |
| **CV AUC (5-fold, dev)** | 0.8213 ± 0.0087 |
| **Threshold** | 0.9004 — Youden's J di validation set |
| **Churn recall / non-churn recall** | 69.0% / 78.7% |
| **Lift — decile risiko terendah** | 40.4% customer kembali vs base rate 10.1% (4.0×) |
| **Leakage `first_order_month`** | ✅ Terdeteksi & diatasi — AUC 0.97 → 0.80 |
| **Leakage waktu lewat fitur lain** | ⚠️ Terdeteksi lewat uji placebo → ✅ diatasi dengan desain snapshot (NB06) |
| **Model v2 — repeat purchase (out-of-time)** | PR-AUC 0.019 vs base 0.0057 (Jun 2018) · signifikan > aturan recency |
| **Model v2 — backtest 3 periode** | RFM LogReg lift top 1% 6.1–6.7× · signifikan > recency di 3/3 periode |
| **Konsistensi notebook ↔ pipeline** | ✅ Angka identik dengan `outputs/metrics.json` |

### Tiga Takeaway Paling Penting

**1. Model yang jujur lebih berharga dari model yang terlihat bagus.**
Leakage ditemukan dalam dua lapis. Lapis pertama — `first_order_month` — membuat AUC 0.97 dan mudah terlihat. Lapis kedua lebih halus: karena label churn ditentukan dari tanggal order terakhir, fitur seperti ongkir dan metode pembayaran ikut membawa informasi waktu. Uji placebo membuktikannya. Setelah label didesain ulang dengan snapshot, ROC-AUC turun ke 0.62 — tetapi kali ini model terbukti menangkap sinyal di luar timing (signifikan mengalahkan aturan recency) dan diuji di periode yang belum pernah dilihat. Model yang lebih rendah tapi valid jauh lebih berharga daripada AUC 0.82 yang mengukur hal yang salah.

**2. Masalah sesungguhnya adalah activation, bukan retention.**
96.9% customer hanya order 1 kali. Loyalty program jangka panjang tidak akan efektif kalau customer tidak pernah datang kembali untuk kedua kalinya. Voucher second-purchase yang diuji lewat A/B test adalah langkah paling berdasar.

**3. Hipotesis ongkir tidak didukung data — dan itu tetap temuan yang berguna.**
Baik EDA maupun SHAP menunjukkan ongkir tinggi tidak berasosiasi dengan churn lebih tinggi. Program gratis ongkir boleh diuji, tetapi harus lewat eksperimen, bukan diasumsikan berhasil.

---

### File Output

```
outputs/
├── model_xgb.pkl                  ← XGBoost pipeline (dari train_pipeline.py) — dipakai API
├── metrics.json                   ← AUC, threshold, confusion matrix, lift, baseline drift
├── metrics_snapshot.json          ← Model v2: metrics out-of-time, grid, uji signifikansi
├── model_snapshot_rfm_logreg.pkl  ← Model v2 yang direkomendasikan
├── model_snapshot_xgboost.pkl     ← Model v2 challenger
├── backtest_snapshot.csv          ← Model v2: metrik per periode backtest
├── backtest_snapshot_significance.csv
├── model.pkl                      ← Random Forest pipeline (dari NB04)
├── model_comparison.csv           ← Perbandingan semua model (pre-leakage fix)
├── model_comparison_final.csv     ← Perbandingan semua model (post-leakage fix)
└── figures/
    ├── roc_curve_all_models.png
    ├── feature_importance_xgb.png
    ├── shap_summary.png
    ├── threshold_tuning.png
    ├── confusion_matrix_comparison.png
    ├── lift_by_decile.png
    ├── placebo_test.png
    ├── snapshot_pr_lift.png
    ├── snapshot_segment_repeat_rate.png
    ├── snapshot_shap_summary.png
    └── snapshot_backtest.png
```

---

*End of Report — Farah Aulia Kirana · Olist Churn Prediction Project*

In [1]:
# Load dan tampilkan model comparison final
import pandas as pd
import os
import joblib

print('=' * 60)
print('  FINAL MODEL COMPARISON')
print('=' * 60)

try:
    df_comp = pd.read_csv('../outputs/model_comparison_final.csv', index_col='Model')
    print(df_comp.to_string())
    print(f'\n🏆 Champion: {df_comp["ROC-AUC"].idxmax()} '
          f'(AUC = {df_comp["ROC-AUC"].max():.4f})')
except FileNotFoundError:
    print('⚠️  Run 04_modeling.ipynb terlebih dahulu untuk generate file ini.')

print()
model_path = '../outputs/model_xgb.pkl'
if os.path.exists(model_path):
    model = joblib.load(model_path)
    print(f'✅ Model loaded: {type(model.named_steps["model"]).__name__}')
    print(f'   Pipeline steps: {list(model.named_steps.keys())}')
else:
    print('⚠️  model_xgb.pkl belum ada — run 04_modeling.ipynb')

  FINAL MODEL COMPARISON
                     ROC-AUC  CV AUC (mean)  CV AUC (std)  F1 Score
Model                                                              
XGBoost               0.8207         0.8213        0.0087    0.8639
Random Forest         0.8005         0.8025        0.0058    0.8800
Decision Tree         0.7748         0.7863        0.0131    0.7697
Logistic Regression   0.6021         0.6122        0.0080    0.6655

🏆 Champion: XGBoost (AUC = 0.8207)



✅ Model loaded: XGBClassifier
   Pipeline steps: ['imputer', 'model']
